#**Assignment 1 BDA**

#**PySpark**

In [1]:
import os       #importing os to set environment variable
def install_java():
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
  !java -version       #check java version
install_java()

!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


openjdk version "1.8.0_462"
OpenJDK Runtime Environment (build 1.8.0_462-8u462-ga~us1-0ubuntu2~22.04.2-b08)
OpenJDK 64-Bit Server VM (build 25.462-b08, mixed mode)
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 384 kB in 2s (178 kB/s)
Reading pa

In [2]:
pip install pyspark

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("Assignment 1 BDA").getOrCreate()

In [5]:
sc = spark.sparkContext

In [6]:
sc.defaultParallelism

2

In [7]:
rdd1 = sc.textFile("/content/drive/MyDrive/Colab Notebooks/BDA Assignments/Data/airlines.csv")

In [8]:
rdd1.count()

85

In [9]:
rdd1.getNumPartitions()

2

In [10]:
for var in rdd1.take(5):
  print(var)

Year,Quarter,Avg_rev_per_seat,booked_seats
1995,1,296.9,46561
1995,2,296.8,37443
1995,3,287.51,34128
1995,4,287.78,30388


In [11]:
header = rdd1.first()

In [12]:
print(header)

Year,Quarter,Avg_rev_per_seat,booked_seats


In [13]:
rdd2 = rdd1.filter(lambda a : a != header)

In [14]:
for var in rdd2.take(5):
  print(var)

1995,1,296.9,46561
1995,2,296.8,37443
1995,3,287.51,34128
1995,4,287.78,30388
1996,1,283.97,47808


In [15]:
# 1. How can you calculate the average revenue per seat for each year and quarter?
rdd3 = rdd2.map(lambda a : (a.split(',')[0] + "-" + a.split(',')[1], round(float(a.split(',')[2]),2)))

In [16]:
for var in rdd3.take(5):
  print(var)

('1995-1', 296.9)
('1995-2', 296.8)
('1995-3', 287.51)
('1995-4', 287.78)
('1996-1', 283.97)


In [17]:
rdd4 = rdd3.reduceByKey(lambda a,b : round(a+b,2))

In [18]:
for var in rdd4.take(5):
  print(var)

('1995-3', 287.51)
('1996-2', 275.78)
('1996-3', 269.49)
('1996-4', 278.33)
('1998-1', 304.74)


In [19]:
sortRDD1 = rdd4.sortByKey()
for var in sortRDD1.take(5):
  print(var)

('1995-1', 296.9)
('1995-2', 296.8)
('1995-3', 287.51)
('1995-4', 287.78)
('1996-1', 283.97)


In [20]:
# 2. What is the PySpark code to find the year and quarter with the highest average revenue per seat?
rddHighestAvgRev = rdd4.sortBy(lambda a : -a[1])
res = rddHighestAvgRev.first()

print(res)

('2014-3', 396.37)


In [21]:
# 3. How can you calculate the total number of booked seats for each year and quarter?
rddBookedSeats = rdd2.map(lambda a : (a.split(',')[0] + "-" + a.split(',')[1], int(a.split(',')[3])))

In [22]:
totalBookedSeatsYQ = rddBookedSeats.reduceByKey(lambda a,b : a+b)
sortedRDD2 = totalBookedSeatsYQ.sortByKey()
for var in sortedRDD2.take(5):
  print(var)

('1995-1', 46561)
('1995-2', 37443)
('1995-3', 34128)
('1995-4', 30388)
('1996-1', 47808)


In [23]:
# 4. What is the PySpark code to determine the year and quarter with the highest total number of booked seats?

highestTotalBookedSeats = totalBookedSeatsYQ.sortBy(lambda a : -a[1])
res1 = highestTotalBookedSeats.first()
print(res1)

('2010-1', 49678)


In [24]:
# 5. How can you calculate the total revenue generated for each year and quarter (revenue = average revenue per seat * total number of booked seats)?

totalRevYQ = rdd2.map(lambda a : (a.split(',')[0] + "-" + a.split(',')[1], round(float(a.split(',')[2]) * int(a.split(',')[3]),2)))

for var in totalRevYQ.take(5):
  print(var)

('1995-1', 13823960.9)
('1995-2', 11113082.4)
('1995-3', 9812141.28)
('1995-4', 8745058.64)
('1996-1', 13576037.76)


In [25]:
# 6. What is the PySpark code to identify the year and quarter with the highest total revenue?
highTotalRevYQ = totalRevYQ.sortBy(lambda a : -a[1])

res2 = highTotalRevYQ.first()
print(res2)

('2014-4', 18819408.48)


In [26]:
# 7. How can you find the average revenue per seat across different years using PySpark?
rdd5 = rdd2.map(lambda a : (a.split(',')[0], float(a.split(',')[2])))
rddYear = rdd5.reduceByKey(lambda a,b : a+b)
sortedRDD3 = rddYear.sortBy(lambda a : -a[1])
for var in sortedRDD3.take(5):
  print(var)

('2014', 1566.8)
('2013', 1528.01)
('2015', 1508.51)
('2012', 1498.7)
('2011', 1454.5300000000002)


In [27]:
# 8. What is the PySpark code to determine the year with the highest average revenue per seat?
res3 = sortedRDD3.first()
print(res3)

('2014', 1566.8)


In [28]:
# 9. How can you calculate the overall average revenue per seat for the entire dataset using PySpark?
rdd5 = rdd2.map(lambda a : float(a.split(',')[2]))
total = rdd5.reduce(lambda a,b : a + b)
print(total)


27698.79


#**PySQL**

In [29]:
# 1. How can you create a database and table in PySQL to store the airline CSV data?
db = spark.read.format("csv").option("header", "true").option("inferschema","true").load("/content/sample_data/Data/airlines.csv")
db.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Avg_rev_per_seat: double (nullable = true)
 |-- booked_seats: integer (nullable = true)



In [30]:
db.show(truncate=False)

+----+-------+----------------+------------+
|Year|Quarter|Avg_rev_per_seat|booked_seats|
+----+-------+----------------+------------+
|1995|1      |296.9           |46561       |
|1995|2      |296.8           |37443       |
|1995|3      |287.51          |34128       |
|1995|4      |287.78          |30388       |
|1996|1      |283.97          |47808       |
|1996|2      |275.78          |43020       |
|1996|3      |269.49          |38952       |
|1996|4      |278.33          |37443       |
|1997|1      |283.4           |35067       |
|1997|2      |289.44          |46565       |
|1997|3      |282.27          |38886       |
|1997|4      |293.51          |37454       |
|1998|1      |304.74          |31315       |
|1998|2      |300.97          |30852       |
|1998|3      |315.25          |38118       |
|1998|4      |316.18          |35393       |
|1999|1      |331.74          |47453       |
|1999|2      |329.34          |38243       |
|1999|3      |317.22          |33048       |
|1999|4   

In [31]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [32]:
db.createOrReplaceTempView("airlines")

In [33]:
# 2. What is the PySQL query to calculate the average revenue per seat for each year and quarter?
q1 = spark.sql("select Year, Quarter, Avg_rev_per_seat as Avg from airlines order by Year, Quarter")
q1.show()

+----+-------+------+
|Year|Quarter|   Avg|
+----+-------+------+
|1995|      1| 296.9|
|1995|      2| 296.8|
|1995|      3|287.51|
|1995|      4|287.78|
|1996|      1|283.97|
|1996|      2|275.78|
|1996|      3|269.49|
|1996|      4|278.33|
|1997|      1| 283.4|
|1997|      2|289.44|
|1997|      3|282.27|
|1997|      4|293.51|
|1998|      1|304.74|
|1998|      2|300.97|
|1998|      3|315.25|
|1998|      4|316.18|
|1999|      1|331.74|
|1999|      2|329.34|
|1999|      3|317.22|
|1999|      4|317.93|
+----+-------+------+
only showing top 20 rows



In [34]:
# 3. How can you find the year and quarter with the highest average revenue per seat using PySQL?
q2 = spark.sql("select Year, Quarter, Avg_rev_per_seat as max_avg from airlines order by Avg_rev_per_seat desc limit 1")
q2.show()

+----+-------+-------+
|Year|Quarter|max_avg|
+----+-------+-------+
|2014|      3| 396.37|
+----+-------+-------+



In [35]:
# 4. What is the PySQL query to calculate the total number of booked seats for each year and quarter?
q3 = spark.sql("select Year, Quarter, booked_seats from airlines")
q3.show()

+----+-------+------------+
|Year|Quarter|booked_seats|
+----+-------+------------+
|1995|      1|       46561|
|1995|      2|       37443|
|1995|      3|       34128|
|1995|      4|       30388|
|1996|      1|       47808|
|1996|      2|       43020|
|1996|      3|       38952|
|1996|      4|       37443|
|1997|      1|       35067|
|1997|      2|       46565|
|1997|      3|       38886|
|1997|      4|       37454|
|1998|      1|       31315|
|1998|      2|       30852|
|1998|      3|       38118|
|1998|      4|       35393|
|1999|      1|       47453|
|1999|      2|       38243|
|1999|      3|       33048|
|1999|      4|       31256|
+----+-------+------------+
only showing top 20 rows



In [36]:
# 5. How can you determine the year and quarter with the highest total number of booked seats using PySQL?
q4 = spark.sql("select Year, Quarter, booked_seats from airlines order by booked_seats desc limit 1")
q4.show()

+----+-------+------------+
|Year|Quarter|booked_seats|
+----+-------+------------+
|2010|      1|       49678|
+----+-------+------------+



In [37]:
# 6. What is the PySQL query to calculate the total revenue generated for each year and quarter (revenue = average revenue per seat * total number of booked seats)?

q5 = spark.sql("select Year, Quarter, round((Avg_rev_per_seat * booked_seats)/1000000,2) as total_rev_in_mill from airlines")
q5.show()

+----+-------+-----------------+
|Year|Quarter|total_rev_in_mill|
+----+-------+-----------------+
|1995|      1|            13.82|
|1995|      2|            11.11|
|1995|      3|             9.81|
|1995|      4|             8.75|
|1996|      1|            13.58|
|1996|      2|            11.86|
|1996|      3|             10.5|
|1996|      4|            10.42|
|1997|      1|             9.94|
|1997|      2|            13.48|
|1997|      3|            10.98|
|1997|      4|            10.99|
|1998|      1|             9.54|
|1998|      2|             9.29|
|1998|      3|            12.02|
|1998|      4|            11.19|
|1999|      1|            15.74|
|1999|      2|            12.59|
|1999|      3|            10.48|
|1999|      4|             9.94|
+----+-------+-----------------+
only showing top 20 rows



In [38]:
# 7. How can you identify the year and quarter with the highest total revenue using PySQL?
q6 = spark.sql("select Year, Quarter, round((Avg_rev_per_seat * booked_seats)/1000000,2) as total_rev_in_mill from airlines order by round((Avg_rev_per_seat * booked_seats)/1000000,2) desc limit 1")
q6.show()

+----+-------+-----------------+
|Year|Quarter|total_rev_in_mill|
+----+-------+-----------------+
|2014|      4|            18.82|
+----+-------+-----------------+



In [39]:
# 8. What is the PySQL query to find the average revenue per seat across different years?
q7 = spark.sql("select Year, round(sum(Avg_rev_per_seat),2) as avg_rev from airlines group by Year order by Year")
q7.show()

+----+-------+
|Year|avg_rev|
+----+-------+
|1995|1168.99|
|1996|1107.57|
|1997|1148.62|
|1998|1237.14|
|1999|1296.23|
|2000|1356.13|
|2001|1279.19|
|2002| 1250.1|
|2003|1261.87|
|2004| 1223.5|
|2005|1228.74|
|2006| 1313.2|
|2007|1300.56|
|2008|1384.63|
|2009|1242.44|
|2010|1343.33|
|2011|1454.53|
|2012| 1498.7|
|2013|1528.01|
|2014| 1566.8|
+----+-------+
only showing top 20 rows



In [40]:
# 9. How can you determine the year with the highest average revenue per seat using PySQL?
q8 = spark.sql("select Year, round(sum(Avg_rev_per_seat),2) as avg_rev from airlines group by Year order by sum(Avg_rev_per_seat) desc limit 1")
q8.show()

+----+-------+
|Year|avg_rev|
+----+-------+
|2014| 1566.8|
+----+-------+



In [41]:
# 10. What is the PySQL query to calculate the overall average revenue per seat for the entire dataset?
q9 = spark.sql("select round(sum(Avg_rev_per_seat),2) as Total_avg_rev from airlines")
q9.show()

+-------------+
|Total_avg_rev|
+-------------+
|     27698.79|
+-------------+



In [42]:
# 1. What is the average revenue per seat for each year and quarter?
q10 = spark.sql("select Year, Quarter, Avg_rev_per_seat as avg_rev from airlines")
q10.show()

+----+-------+-------+
|Year|Quarter|avg_rev|
+----+-------+-------+
|1995|      1|  296.9|
|1995|      2|  296.8|
|1995|      3| 287.51|
|1995|      4| 287.78|
|1996|      1| 283.97|
|1996|      2| 275.78|
|1996|      3| 269.49|
|1996|      4| 278.33|
|1997|      1|  283.4|
|1997|      2| 289.44|
|1997|      3| 282.27|
|1997|      4| 293.51|
|1998|      1| 304.74|
|1998|      2| 300.97|
|1998|      3| 315.25|
|1998|      4| 316.18|
|1999|      1| 331.74|
|1999|      2| 329.34|
|1999|      3| 317.22|
|1999|      4| 317.93|
+----+-------+-------+
only showing top 20 rows



In [43]:
# 2. Which year and quarter had the highest average revenue per seat?
q11 = spark.sql("select Year, Quarter , Avg_rev_per_seat from airlines order by Avg_rev_per_seat desc limit 1")
q11.show()

+----+-------+----------------+
|Year|Quarter|Avg_rev_per_seat|
+----+-------+----------------+
|2014|      3|          396.37|
+----+-------+----------------+



In [44]:
# 3. How does the average revenue per seat vary across different quarters within a year?
q12 = spark.sql("select Quarter, round(avg(Avg_rev_per_seat),2) as avg_seasonal_ARPS from airlines group by Quarter order by Quarter")
q12.show()

+-------+-----------------+
|Quarter|avg_seasonal_ARPS|
+-------+-----------------+
|      1|           330.61|
|      2|           332.34|
|      3|           327.56|
|      4|           328.48|
+-------+-----------------+



In [45]:
# 4. What is the total number of booked seats for each year and quarter?
q13 = spark.sql("select Year, Quarter, booked_seats from airlines order by Year, Quarter")
q13.show()

+----+-------+------------+
|Year|Quarter|booked_seats|
+----+-------+------------+
|1995|      1|       46561|
|1995|      2|       37443|
|1995|      3|       34128|
|1995|      4|       30388|
|1996|      1|       47808|
|1996|      2|       43020|
|1996|      3|       38952|
|1996|      4|       37443|
|1997|      1|       35067|
|1997|      2|       46565|
|1997|      3|       38886|
|1997|      4|       37454|
|1998|      1|       31315|
|1998|      2|       30852|
|1998|      3|       38118|
|1998|      4|       35393|
|1999|      1|       47453|
|1999|      2|       38243|
|1999|      3|       33048|
|1999|      4|       31256|
+----+-------+------------+
only showing top 20 rows



In [46]:
# 5. Which year and quarter had the highest total number of booked seats?
q14 = spark.sql("select Year, Quarter, booked_seats from airlines order by booked_seats desc limit 1")
q14.show()

+----+-------+------------+
|Year|Quarter|booked_seats|
+----+-------+------------+
|2010|      1|       49678|
+----+-------+------------+



In [47]:
# 6. How does the total number of booked seats vary across different quarters within a year?
q15 = spark.sql("select Quarter, sum(booked_seats) as avg_seats_quarter from airlines group by Quarter order by sum(booked_seats) desc")
q15.show()

+-------+-----------------+
|Quarter|avg_seats_quarter|
+-------+-----------------+
|      1|           873761|
|      3|           827111|
|      4|           821351|
|      2|           807596|
+-------+-----------------+



In [48]:
# 7. What is the total revenue generated for each year and quarter (revenue = average revenue per seat * total number of booked seats)?
q16 = spark.sql("select Year, Quarter, round((Avg_rev_per_seat * booked_seats)/1000000,2) as total_rev_in_mill from airlines")
q16.show()

+----+-------+-----------------+
|Year|Quarter|total_rev_in_mill|
+----+-------+-----------------+
|1995|      1|            13.82|
|1995|      2|            11.11|
|1995|      3|             9.81|
|1995|      4|             8.75|
|1996|      1|            13.58|
|1996|      2|            11.86|
|1996|      3|             10.5|
|1996|      4|            10.42|
|1997|      1|             9.94|
|1997|      2|            13.48|
|1997|      3|            10.98|
|1997|      4|            10.99|
|1998|      1|             9.54|
|1998|      2|             9.29|
|1998|      3|            12.02|
|1998|      4|            11.19|
|1999|      1|            15.74|
|1999|      2|            12.59|
|1999|      3|            10.48|
|1999|      4|             9.94|
+----+-------+-----------------+
only showing top 20 rows



In [49]:
# 8. Which year and quarter had the highest total revenue?
q17 = spark.sql("select Year, Quarter, round((Avg_rev_per_seat * booked_seats)/1000000,2) as total_rev_in_mill from airlines order by total_rev_in_mill desc limit 1")
q17.show()

+----+-------+-----------------+
|Year|Quarter|total_rev_in_mill|
+----+-------+-----------------+
|2014|      4|            18.82|
+----+-------+-----------------+



In [50]:
# 9. How does the total revenue vary across different quarters within a year?
q18 = spark.sql("select Quarter, round(sum(Avg_rev_per_seat * booked_seats)/1000000,2) as total_rev_in_mill from airlines group by Quarter order by Quarter")
q18.show()

+-------+-----------------+
|Quarter|total_rev_in_mill|
+-------+-----------------+
|      1|           288.86|
|      2|           268.33|
|      3|           271.97|
|      4|           270.99|
+-------+-----------------+



In [51]:
# 10. Can you identify any trends or patterns in the revenue or number of booked seats over the years?
q19 = spark.sql("select Year, round(sum(Avg_rev_per_seat * booked_seats)/1000000,2) as total_annual_rev_millions, sum(booked_seats) as total_annual_booked_seats from airlines group by Year order by total_annual_rev_millions desc")
q19.show()

+----+-------------------------+-------------------------+
|Year|total_annual_rev_millions|total_annual_booked_seats|
+----+-------------------------+-------------------------+
|2013|                    66.36|                   173676|
|2014|                    62.62|                   159823|
|2015|                    62.38|                   165438|
|2012|                     62.2|                   166076|
|2008|                    57.65|                   166897|
|2007|                    57.31|                   176299|
|2001|                    55.53|                   173598|
|2010|                    54.86|                   163741|
|2000|                    52.34|                   154376|
|2011|                    51.89|                   142647|
|2004|                    50.63|                   164800|
|2006|                    50.44|                   153789|
|2003|                    49.27|                   156153|
|1999|                    48.76|                   15000

In [52]:
# 11. How does the average revenue per seat vary across different years?
q20 = spark.sql("select Year, round(sum(Avg_rev_per_seat),2) as annual_ARPS from airlines group by Year order by annual_ARPS desc")
q20.show()

+----+-----------+
|Year|annual_ARPS|
+----+-----------+
|2014|     1566.8|
|2013|    1528.01|
|2015|    1508.51|
|2012|     1498.7|
|2011|    1454.53|
|2008|    1384.63|
|2000|    1356.13|
|2010|    1343.33|
|2006|     1313.2|
|2007|    1300.56|
|1999|    1296.23|
|2001|    1279.19|
|2003|    1261.87|
|2002|     1250.1|
|2009|    1242.44|
|1998|    1237.14|
|2005|    1228.74|
|2004|     1223.5|
|1995|    1168.99|
|1997|    1148.62|
+----+-----------+
only showing top 20 rows



In [53]:
# 12. Which year had the highest average revenue per seat?
q21 = spark.sql("select Year, Avg_rev_per_seat from airlines order by Avg_rev_per_seat desc limit 1")
q21.show()

+----+----------------+
|Year|Avg_rev_per_seat|
+----+----------------+
|2014|          396.37|
+----+----------------+



In [54]:
# 13. What is the overall average revenue per seat for the entire dataset?
q22 = spark.sql("select round(sum(Avg_rev_per_seat * booked_seats)/sum(booked_seats),2) as overall_ARPS from airlines")
q22.show()

+------------+
|overall_ARPS|
+------------+
|      330.39|
+------------+



In [55]:
# 14. Can you identify any seasonal trends in the average revenue per seat?
q23 = spark.sql("select Quarter, round(avg(Avg_rev_per_seat),2) as avg_seasonal_ARPS from airlines group by Quarter order by Quarter")
q23.show()

+-------+-----------------+
|Quarter|avg_seasonal_ARPS|
+-------+-----------------+
|      1|           330.61|
|      2|           332.34|
|      3|           327.56|
|      4|           328.48|
+-------+-----------------+



In [56]:
# 15. How does the total number of booked seats vary across different years?
q24 = spark.sql("select Year, sum(booked_seats) as annual_BST from airlines group by Year")
q24.show()

+----+----------+
|Year|annual_BST|
+----+----------+
|2003|    156153|
|2007|    176299|
|2015|    165438|
|2006|    153789|
|2013|    173676|
|1997|    157972|
|2014|    159823|
|2004|    164800|
|1996|    167223|
|1998|    135678|
|2012|    166076|
|2009|    150308|
|1995|    148520|
|2001|    173598|
|2005|    150610|
|2000|    154376|
|2010|    163741|
|2011|    142647|
|2008|    166897|
|1999|    150000|
+----+----------+
only showing top 20 rows



In [57]:
# 16. Which year had the highest total number of booked seats?
q25 = spark.sql("select Year, sum(booked_seats) as annual_BST from airlines group by Year order by sum(booked_seats) desc limit 1")
q25.show()

+----+----------+
|Year|annual_BST|
+----+----------+
|2007|    176299|
+----+----------+



In [58]:
# 17. What is the overall total number of booked seats for the entire dataset?
q26 = spark.sql("select sum(booked_seats) as overall_total_BS from airlines")
q26.show()

+----------------+
|overall_total_BS|
+----------------+
|         3329819|
+----------------+



In [59]:
# 18. Can you identify any seasonal trends in the total number of booked seats?
q27 = spark.sql("select Quarter, sum(booked_seats) as seasonal_BST from airlines group by Quarter order by sum(booked_seats) desc")
q27.show()

+-------+------------+
|Quarter|seasonal_BST|
+-------+------------+
|      1|      873761|
|      3|      827111|
|      4|      821351|
|      2|      807596|
+-------+------------+



In [60]:
# 19. How does the total revenue generated vary across different years?
q28 = spark.sql("select Year, round(sum(Avg_rev_per_seat * booked_seats)/1000000,2) as total_annual_rev_millions from airlines group by Year order by total_annual_rev_millions desc")
q28.show()

+----+-------------------------+
|Year|total_annual_rev_millions|
+----+-------------------------+
|2013|                    66.36|
|2014|                    62.62|
|2015|                    62.38|
|2012|                     62.2|
|2008|                    57.65|
|2007|                    57.31|
|2001|                    55.53|
|2010|                    54.86|
|2000|                    52.34|
|2011|                    51.89|
|2004|                    50.63|
|2006|                    50.44|
|2003|                    49.27|
|1999|                    48.76|
|2002|                     47.5|
|2009|                    46.75|
|2005|                    46.38|
|1996|                    46.36|
|1997|                    45.39|
|1995|                    43.49|
+----+-------------------------+
only showing top 20 rows



In [61]:
# 20. Which year had the highest total revenue generated?
# 19. How does the total revenue generated vary across different years?
q29 = spark.sql("select Year, round(sum(Avg_rev_per_seat * booked_seats)/1000000,2) as total_annual_rev_millions from airlines group by Year order by total_annual_rev_millions desc limit 1")
q29.show()

+----+-------------------------+
|Year|total_annual_rev_millions|
+----+-------------------------+
|2013|                    66.36|
+----+-------------------------+

